# Investigating the effect of different kernel size on SAL

SAL is particularly useful in smoothing nuisance variables (which is one kind of transformation). Given a different kernel sizes (receptive fields are another kind of transformation), the goal is to understand how SAL behaves and it's impact in understanding an image <br>

**Goal:** Quantify how convolutional kernel size (which changes the model’s effective receptive field) interacts with SAL-style local marginalization (anti-aliased downsampling) to shape the invariance–selectivity trade-off under small nuisance transformations (tiny shifts/rotations/scale) and occlusion. <br>

**What will be tested:** Three matched-capacity CNNs (3×3, 5×5, 7×7 kernels) × two pooling modes (standard vs. anti-aliased/SAL-ish). We’ll measure:
* Feature stability under small group actions $g$ (tiny translation/rotation/scale).
* Near-miss accuracy on confusable shape pairs.
* Occlusion sensitivity vs occlusion % and location.

## Dataset + Pre-processing

I am using the Kaggle dataset of geometric shapes [linked here](https://www.kaggle.com/datasets/reevald/geometric-shapes-mathematics).

In [2]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image, ImageDraw
import math, random

Data Loading

In [4]:
IMG_SIZE = 224 # staying at native resolution (to avoid additional transformations)

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # applies the nuisance transformations
    transforms.RandomAffine(
        degrees = 5,
        translate = (0.05, 0.05),
        scale = (0.95, 1.05)
    ),
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_dir = "dataset/train"
test_dir = "dataset/test"
val_dir = "dataset/val"

train_data = datasets.ImageFolder(
    root = train_dir,
    transform = train_transforms
)
test_data = datasets.ImageFolder(
    root = test_dir,
    transform = test_transforms
)
val_data = datasets.ImageFolder(
    root = val_dir,
    transform = test_transforms
)

# Creating DataLoaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2)

print("Classes: ", train_data.classes)
print("Number of training samples: ", len(train_data))

Classes:  ['circle', 'kite', 'parallelogram', 'rectangle', 'rhombus', 'square', 'trapezoid', 'triangle']
Number of training samples:  12000
